In [2]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3
import os

I0000 00:00:1782821635.399739   40552 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Segmentation Models: using `tf.keras` framework.


In [3]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet , mergeTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet  , mergeTestSet  = tts.dataTensorLoading(testSet)


In [ ]:
src = "data/converted"
pname = os.listdir(src)
cnt = 0
for name in pname :
    res = tts.merge(os.path.join(src , name))
    cnt+=1
else :
    print(f"number of merged :" , cnt)

In [4]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet , mergeTrainSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet , mergeTestSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")

data/CTA nii/AHMU1218096/cta_images_head_AHMU1218096.nii.gz
data/CTA nii/AHMU1218096/3D_aneurysm_AHMU1218096_@32_@MCA-label.nii
NoMerge
data/CTA nii/990784-TAHEREHM...BAZAN/990784-TAHEREHM...BAZAN_Brain_-CTA_20171010132707_4.nii
data/CTA nii/990784-TAHEREHM...BAZAN/990784-TAHEREHM...BAZAN_Brain_-CTA_20171010132707_4_@29_@BASILAR-label.nii
NoMerge
data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4.nii
data/CTA nii/1009194-AHMADVAHHABI/1009194-AHMADVAHHABI_Brain_-CTA_20180329122737_4_@37_@MCA-label.nii
NoMerge
data/CTA nii/1121445-AMENEKHAVARI/1121445-AMENEKHAVARI_CTA_20211120080858_4.nii
data/CTA nii/1121445-AMENEKHAVARI/1121445-AMENEKHAVARI_CTA_20211120080858_4_@16_@ACOM-label.nii
NoMerge
data/CTA nii/1017563-KHADIJE...JANDI/1017563-KHADIJE...JANDI_Brain_-CTA_20180703064010_4.nii
data/CTA nii/1017563-KHADIJE...JANDI/1017563-KHADIJE...JANDI_Brain_-CTA_20180703064010_4_@46_@DACA-label.nii
NoMerge
data/CTA nii/AHMU1218093/cta_images_head_AHMU1218093.nii.gz


In [5]:
# pipeline configuring

geo      = utl.randomGeo(p=0.7)
crop     = utl.volume_crop((128 , 128 , 128))
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 128 , 128 , 128 , 3] ,
    labelShape=[None , 128 , 128 , 128 , 1]
)

windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=0.7 ,
    p_ww=0.7
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64 , tf.float64])
def rimg(imgPath , labelPath , mergePath) :
    return utl.read_img(imgPath , labelPath , mergePath)
def read_img(img , label , merge) :
    imglbl = rimg(img , label , merge)
    img   = imglbl[0]
    label = imglbl[1]
    merge = imglbl[2]
    return img , label , merge

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



I0000 00:00:1782821644.961664   40552 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1782821645.129071   40552 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1782821645.140414   40552 read_numa_node.cc:68] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node so this will  be massaged to NUMA node zero in some places. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1

In [6]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet , mergeTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    .cache("myCacheTrain")
    .shuffle(buffer_size=120 , seed=42 , reshuffle_each_iteration=True)
    
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        geo.flipX ,
        num_parallel_calls=4
    )
    .map(
        geo.flipY ,
        num_parallel_calls=4
    )
    .map(
        geo.flipZ ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )

    .batch(batch_size=4)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTestSet , labelTestSet , mergeTestSet))

    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )
    .map(
        utl.channelize
    )
    .cache("myCacheValid")
    .batch(batch_size=2)
    .map(
        windower.apply_default , 
        num_parallel_calls=4
    )
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

In [ ]:
for data in dataloaderValid.take(5) :
    print("image shape :" , data[0].shape)
    print("label shape :" , data[1].shape)

In [7]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4
)

# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze34_border , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

In [ ]:
print(model.summary())

In [8]:
# compilation
lr = keras.optimizers.schedules.PiecewiseConstantDecay(
    [
        5530 ,
        11060 ,

    ] ,
    [
        1e-3 ,
        1e-4 ,
        1e-5
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice
    ] ,
)

In [9]:
logger = keras.callbacks.CSVLogger("tuning_logs/testRun_2.csv")

# model training
history = model.fit(
    x = dataloaderTrain ,
    epochs=250 ,
    validation_data = dataloaderValid ,
    callbacks=[
        logger
    ]

)

Epoch 1/250


I0000 00:00:1782821686.888438   40693 service.cc:153] XLA service 0x756a680770f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782821686.888508   40693 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3060, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1782821687.668795   40693 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1782821692.131754   40693 cuda_dnn.cc:461] Loaded cuDNN version 92300
E0000 00:00:1782821695.524545   40693 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1782821718.572704   40693 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - dice: 8.4840e-05 - loss: 0.2875 - v__recall: 13.4492

E0000 00:00:1782821896.320227   40692 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1782821905.658402   40692 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1782821922.190269   40691 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1782821925.780588   40691 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


79/79 ━━━━━━━━━━━━━━━━━━━━ 264s 2s/step - dice: 8.6034e-05 - loss: 0.2867 - v__recall: 13.3238 - val_dice: 0.0000e+00 - val_loss: 0.2059 - val_v__recall: 0.0000e+00
Epoch 2/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 115s 1s/step - dice: 0.0861 - loss: 0.1991 - v__recall: 9.7994 - val_dice: 0.0000e+00 - val_loss: 0.2036 - val_v__recall: 0.0000e+00
Epoch 3/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - dice: 0.2704 - loss: 0.1894 - v__recall: 44.5242 - val_dice: 0.0000e+00 - val_loss: 0.2036 - val_v__recall: 0.0000e+00
Epoch 4/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - dice: 0.3657 - loss: 0.1334 - v__recall: 50.2797 - val_dice: 0.0000e+00 - val_loss: 0.2041 - val_v__recall: 0.0000e+00
Epoch 5/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - dice: 0.5497 - loss: 0.0853 - v__recall: 61.9997 - val_dice: 0.0000e+00 - val_loss: 0.2029 - val_v__recall: 0.0000e+00
Epoch 6/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - dice: 0.5524 - loss: 0.0842 - v__recall: 61.2087 - val_dice: 0.0000e+00 - val_loss: 0.2040

I0000 00:00:1782823708.074274   44147 shuffle_dataset_op.cc:453] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 80 of 120
I0000 00:00:1782823713.218845   44147 shuffle_dataset_op.cc:483] Shuffle buffer filled.


79/79 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - dice: 0.6502 - loss: 0.0625 - v__recall: 67.8336 - val_dice: 0.6946 - val_loss: 0.0604 - val_v__recall: 68.4826
Epoch 18/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 119s 1s/step - dice: 0.6558 - loss: 0.0627 - v__recall: 70.1894 - val_dice: 0.6679 - val_loss: 0.0645 - val_v__recall: 61.4036
Epoch 19/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - dice: 0.6594 - loss: 0.0622 - v__recall: 68.8928 - val_dice: 0.6971 - val_loss: 0.0588 - val_v__recall: 67.9826
Epoch 20/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - dice: 0.6312 - loss: 0.0688 - v__recall: 67.8668 - val_dice: 0.6765 - val_loss: 0.0621 - val_v__recall: 61.5638
Epoch 21/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - dice: 0.6556 - loss: 0.0656 - v__recall: 69.1765 - val_dice: 0.6556 - val_loss: 0.0659 - val_v__recall: 58.3714
Epoch 22/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 113s 1s/step - dice: 0.6646 - loss: 0.0621 - v__recall: 69.8399 - val_dice: 0.6336 - val_loss: 0.0698 - val_v__recall: 54.0600
Epoch 23/25

I0000 00:00:1782829835.427008   48279 shuffle_dataset_op.cc:453] ShuffleDatasetV3:4: Filling up shuffle buffer (this may take a while): 119 of 120
I0000 00:00:1782829835.787105   48279 shuffle_dataset_op.cc:483] Shuffle buffer filled.


79/79 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - dice: 0.7156 - loss: 0.0499 - v__recall: 73.0566 - val_dice: 0.7049 - val_loss: 0.0578 - val_v__recall: 67.2159
Epoch 69/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - dice: 0.7159 - loss: 0.0514 - v__recall: 73.7396 - val_dice: 0.7042 - val_loss: 0.0600 - val_v__recall: 70.2229
Epoch 70/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - dice: 0.6973 - loss: 0.0560 - v__recall: 72.1237 - val_dice: 0.7202 - val_loss: 0.0546 - val_v__recall: 68.0906
Epoch 71/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - dice: 0.7101 - loss: 0.0525 - v__recall: 72.6724 - val_dice: 0.7298 - val_loss: 0.0536 - val_v__recall: 71.3246
Epoch 72/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - dice: 0.7200 - loss: 0.0495 - v__recall: 74.7599 - val_dice: 0.7351 - val_loss: 0.0528 - val_v__recall: 72.9050
Epoch 73/250
79/79 ━━━━━━━━━━━━━━━━━━━━ 118s 1s/step - dice: 0.7265 - loss: 0.0511 - v__recall: 76.3582 - val_dice: 0.7373 - val_loss: 0.0526 - val_v__recall: 73.6027
Epoch 74/25